In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict


In [24]:
import pandas as pd

df = pd.read_csv("C:\\Users\\koolr\\Downloads\\archive (3)\\results.csv")
df.head()


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [25]:
df.shape

(49477, 9)

In [26]:
df.describe

<bound method NDFrame.describe of              date home_team   away_team  home_score  away_score  \
0      1872-11-30  Scotland     England         0.0         0.0   
1      1873-03-08   England    Scotland         4.0         2.0   
2      1874-03-07  Scotland     England         2.0         1.0   
3      1875-03-06   England    Scotland         2.0         2.0   
4      1876-03-04  Scotland     England         3.0         0.0   
...           ...       ...         ...         ...         ...   
49472  2026-06-27    Jordan   Argentina         NaN         NaN   
49473  2026-06-27  Colombia    Portugal         NaN         NaN   
49474  2026-06-27  DR Congo  Uzbekistan         NaN         NaN   
49475  2026-06-27    Panama     England         NaN         NaN   
49476  2026-06-27   Croatia       Ghana         NaN         NaN   

           tournament             city        country  neutral  
0            Friendly          Glasgow       Scotland    False  
1            Friendly          

In [27]:
print(df.columns.tolist())

['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']


In [28]:
print(df["date"].max())
print(df["date"].min())

2026-06-27
1872-11-30


In [29]:
df["date"]= pd.to_datetime(df["date"])
df = df[(df['date'] >= '2010-01-01') & (df['date'] < '2026-06-11')].sort_values('date').reset_index(drop=True)

print(df.shape)
print(df.head())

(15817, 9)
        date home_team    away_team  home_score  away_score tournament  \
0 2010-01-02      Iran  North Korea         1.0         0.0   Friendly   
1 2010-01-02     Qatar         Mali         0.0         0.0   Friendly   
2 2010-01-02     Syria     Zimbabwe         6.0         0.0   Friendly   
3 2010-01-02     Yemen   Tajikistan         0.0         1.0   Friendly   
4 2010-01-03    Angola       Gambia         1.0         1.0   Friendly   

                         city   country  neutral  
0                        Doha     Qatar     True  
1                        Doha     Qatar    False  
2                Kuala Lumpur  Malaysia     True  
3                      Sana'a     Yemen    False  
4  Vila Real de Santo António  Portugal     True  


In [30]:
def expected(ra, rb):
    return 1/(1+10**((rb-ra)/400))

def update_elo(ra, rb, result, k=30):
    e= expected(ra,rb)
    new_ra= ra+k*(result-e)
    new_rb= rb+k*((1-result)-(1-e))
    return new_ra, new_rb

elo= defaultdict(lambda:1500)

for _, row in df.iterrows():
    h, a= row["home_team"], row["away_team"]
    if row ["home_score"]>row["away_score"]:
        result=1
    elif row ["home_score"]<row["away_score"]:
        result=0
    else:
        result=0.5

    elo[h], elo[a]=update_elo(elo[h], elo[a], result)

elo_df= pd.DataFrame(elo.items(), columns=["teams", "elo"])
elo_df= elo_df.sort_values("elo", ascending= False).reset_index(drop=True)
elo_df["rank"]= elo_df.index+ 1
elo_df["elo"]= elo_df["elo"].round(2)
print(elo_df.head(48))

             teams      elo  rank
0        Argentina  1955.62     1
1            Spain  1948.68     2
2           France  1890.57     3
3          Morocco  1878.77     4
4           Brazil  1865.52     5
5            Japan  1862.70     6
6         Portugal  1862.46     7
7          England  1853.22     8
8         Colombia  1847.46     9
9          Germany  1846.94    10
10     Netherlands  1823.84    11
11          Mexico  1816.06    12
12         Ecuador  1814.82    13
13         Algeria  1808.67    14
14            Iran  1803.05    15
15         Croatia  1792.02    16
16         Senegal  1791.27    17
17           Italy  1790.94    18
18         Belgium  1790.32    19
19     South Korea  1786.27    20
20         Uruguay  1774.90    21
21          Turkey  1772.60    22
22          Canada  1763.28    23
23       Australia  1760.66    24
24     Switzerland  1759.08    25
25         Denmark  1758.99    26
26   United States  1754.15    27
27         Austria  1749.16    28
28          No

In [31]:
import os
os.makedirs("data", exist_ok=True)
elo_df.to_csv("data/elo_ratings.csv", index=False)
print("ELO ratings saved!")


ELO ratings saved!


In [32]:
wc_check = df[df['tournament'] == 'FIFA World Cup']
print(wc_check[wc_check['date'] >= '2026-06-11'])

Empty DataFrame
Columns: [date, home_team, away_team, home_score, away_score, tournament, city, country, neutral]
Index: []


In [33]:
print(df.shape)
print(df['date'].max())

(15817, 9)
2026-06-10 00:00:00
